# **Atividade Prática**
<font size=3>

- **Tema:** amostragem e fluxo de trabalho.
- **Prazo de entrega:** 30 de Abril.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSeHK5qrS-_wttHK6XXKBK2-SXv1nxU3Xhk8x__eP1ZrHulHDw/viewform?usp=header).

---

### **1. Questão:**
<font size=3>

Com base no *dataset* de [Fraudes de cartões de crédito](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** a proporção de classes;
2. Faça um divisão estratificada de dados em treinamento, validação e teste;
3. Faça a busca do hiperparâmetro $k$, **em um laço** `for`, do modelo **k-NN**. Utilize os dados de validação para medir a performance do modelo a cada valor de $k$;
4. **Imprima na tela** o melhor valor de $k$ e **retreine** o modelo com este valor. Utilize os dados de treinamento + validação para o *fit* do modelo;
5. Faça a avaliação final do melhor modelo com os dados de teste.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# from google.colab import drive
# drive.mount('/content/drive') 

dataset = pd.read_csv("/content/drive/MyDrive/5 semestre /Aprendizado de maquina supervisionado/IMD3002 - Aprendizado de Máquina Supervisionado/2-unidade/dataset/credit_card_fraud.csv")
dataset.head()
scaler = StandardScaler()

x = dataset.drop(columns=['Class'])
y = dataset['Class']
x_train, x_dev, y_train, y_dev = train_test_split(x,y , test_size=0.3, random_state=42, stratify=y )
x_train_scaled = scaler.fit_transform(x_train)


print(f"Proporção da classe 1 no treino: {y_train.mean():.2f}")
print(f"Proporção da classe 1 no teste: {y_dev.mean():.2f}\n")

x_val, x_test, y_val, y_test = train_test_split(x_dev, y_dev, test_size=0.5, stratify=y_dev,random_state=42) 

x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

print(f"X-train:{x_train.shape}, X-val:{x_val_scaled.shape}, X-test:{x_test_scaled.shape}")
print(f"y-train:{y_train.shape}, y-val:{y_val.shape}, y-test:{y_test.shape}")


print(f"Proporção da classe 1 no treino: {y_train.mean():.2f}")
print(f"Proporção da classe 1 na validation: {y_val.mean():.2f}")
print(f"Proporção da classe 1 no teste: {y_test.mean():.2f}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proporção da classe 1 no treino: 0.15
Proporção da classe 1 no teste: 0.15

X-train:(2334, 32), X-val:(500, 32), X-test:(501, 32)
y-train:(2334,), y-val:(500,), y-test:(501,)
Proporção da classe 1 no treino: 0.15
Proporção da classe 1 na validation: 0.15
Proporção da classe 1 no teste: 0.15


In [103]:

scores = []

for k in range(1, 32):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(x_train_scaled, y_train)

    y_pred_val = model.predict(x_val_scaled)

    acc = accuracy_score(y_val, y_pred_val)

    scores.append(acc)

best_k = np.argmax(scores) + 1
print(f"Melhor k (validação): {best_k}")

final_model = KNeighborsClassifier(n_neighbors=best_k)

final_model.fit(np.concatenate([x_train_scaled, x_val_scaled]),
                np.concatenate([y_train, y_val]))

y_pred = final_model.predict(x_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f"Acurácia no conjunto de teste: {acc:.2f}")

Melhor k (validação): 6
Acurácia no conjunto de teste: 0.95


### **2. Questão:**
<font size=3>

Com base no *dataset* de [Pinguins](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data?select=penguins_size.csv), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** todas as classes das variáveis categóricas **e** suas proporções;
2. Caso exista alguma **classe indefinida**, veja quantas amostras esta classe apresenta. Caso seja um valor pequeno de amostras, remova a classe indefinida;
3. Observe se o *dataframe* apresenta valores `NaN`. Caso existam, remova a linha correspondente do *dataframe*;
4. Defina o atributo `species` como variável alvo (`y`), e as demais como `X`;
5. Realize as transformação necessárias nos dados categóricos e numéricos com base nas classes [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) e [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html);
6. Utilize a abordagem correta de treinamento e avaliação do modelo de **Regressão Logística** com base no tamanho do *dataset* e a proporção de classes.
   

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder 
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline


dataset = pd.read_csv("/content/drive/MyDrive/5 semestre /Aprendizado de maquina supervisionado/IMD3002 - Aprendizado de Máquina Supervisionado/2-unidade/dataset/penguins_size.csv")
dataset.head()

dataset = dataset[dataset['sex'] != '.']
dataset = dataset.dropna() 

x = dataset.drop(columns=["species"])
y = dataset["species"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

numeric_atribut = ["culmen_length_mm","culmen_depth_mm","flipper_length_mm","body_mass_g"]
category_atribut = ["island","sex"]

ohec = OneHotEncoder()

preprocesamento = ColumnTransformer(transformers=[("num",scaler,numeric_atribut),("cat",ohec,category_atribut),],remainder='passthrough')
preprocesamento
x_train_pro = preprocesamento.fit_transform(x_train)
x_test_pro = preprocesamento.transform(x_test)



pipe = Pipeline(steps=[("preprocesamento",preprocesamento),("classifier",LogisticRegression())])
pipe

Pipeline(steps=[('preprocesamento',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['culmen_length_mm',
                                                   'culmen_depth_mm',
                                                   'flipper_length_mm',
                                                   'body_mass_g']),
                                                 ('cat', OneHotEncoder(),
                                                  ['island', 'sex'])])),
                ('classifier', LogisticRegression())])

In [105]:
pipe.fit(x_train, y_train)

print(f"Acurácia no Treino: {pipe.score(x_train, y_train):.2%}")
print(f"Acurácia no Teste: {pipe.score(x_test, y_test):.2%}")

y_pred = pipe.predict(x_test)
acc = accuracy_score(y_test, y_pred)
print(f"Acurácia Real: {acc:.2f}")

Acurácia no Treino: 99.62%
Acurácia no Teste: 98.51%
Acurácia Real: 0.99


### **3. Questão:**
<font size=3>

Com base no *dataset* de [Cogumelos](https://www.kaggle.com/datasets/uciml/mushroom-classification), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** a proporção da classificação definidas aos cogumelos (`class`);
2. Defina o atributo `class` como variável alvo (`y`), e as demais como `X`;
3. Faça a divisão dos *dataset* entre **treinamento** e **teste**;
4. Realize as transformação necessárias nos dados categóricos com base nas classes [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) e [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html);
5. Defina um objeto `Pipeline` para encadear o pré-processamento de `X_train` junto ao treinamento do modelo **k-NN**;
6. Utilize a classe [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) para realizar a busca do melhor par de hiperparâmetros `n_neighbors` e `p` . Para esta busca, defina 20 iterações e 3 divisões para a validação cruzada;
7. Faça a previsão e avaliação do melhor modelo com os dados de teste .
   

In [ ]:

dataset = pd.read_csv("/content/drive/MyDrive/5 semestre /Aprendizado de maquina supervisionado/IMD3002 - Aprendizado de Máquina Supervisionado/2-unidade/dataset/mushrooms.csv")
dataset.head()


x = dataset.drop(columns=["class"])
y = dataset["class"]

y_encoded = le.fit_transform(y) 

x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)


category_atribut = x.columns.tolist()

ohec = OneHotEncoder(handle_unknown='ignore')

preprocesamento = ColumnTransformer(transformers=[("cat",ohec,category_atribut),],remainder='passthrough')
preprocesamento
x_train_pro = preprocesamento.fit_transform(x_train)

pipe = Pipeline(steps=[('preprocessor', preprocesamento),
                       ('classifier', KNeighborsClassifier())])

pipe


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['cap-shape', 'cap-surface',
                                                   'cap-color', 'bruises',
                                                   'odor', 'gill-attachment',
                                                   'gill-spacing', 'gill-size',
                                                   'gill-color', 'stalk-shape',
                                                   'stalk-root',
                                                   'stalk-surface-above-ring',
                                                   'stalk-surface-below-ring',
                                                   'stalk-color-above-ring',
                                                   'stalk-color-below-ring',
                                                   'veil-type', 'veil-color',
                                                   'ring-number', 'ring-type',
                                                   'spore-print-color',
                                                   'population',
                                                   'habitat'])])),
                ('classifier', KNeighborsClassifier())])

In [107]:
pipe.fit(x_train, y_train)

print(f"Acurácia no Treino: {pipe.score(x_train, y_train):.2%}")
print(f"Acurácia no Teste: {pipe.score(x_test, y_test):.2%}")

y_pred = pipe.predict(x_test)
acc = accuracy_score(y_test, y_pred)
print(f"Acurácia Real: {acc:.2f}")

Acurácia no Treino: 100.00%
Acurácia no Teste: 100.00%
Acurácia Real: 1.00


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {'classifier__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'classifier__p': [1, 2]
             }

random_search = RandomizedSearchCV(
    estimator=pipe,          
    param_distributions=param_grid  ,
    n_iter=20,                   
    cv=3,                        
    scoring='accuracy',
    verbose=1,
    random_state=42
)

random_search.fit(x_train, y_train)

print(f"\nMelhores parâmetros: {random_search.best_params_}")
print(f"Melhor score de acurácia (validação cruzada): {random_search.best_score_:.4f}")
y_pred_final = random_search.predict(x_test)
acc_final = accuracy_score(y_test, y_pred_final)

print(f"Acurácia final no Teste com o melhor k-NN: {acc_final:.2f}")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 14 is smaller than n_iter=20. Running 14 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 14 candidates, totalling 42 fits

Melhores parâmetros: {'classifier__p': 1, 'classifier__n_neighbors': 3}
Melhor score de acurácia (validação cruzada): 0.9992
Acurácia final no Teste com o melhor k-NN: 1.00
